# Modeling — Loan Approval Prediction

Trains and tunes three classifiers — Logistic Regression, Random Forest, and XGBoost — on the processed split from `02_Data_Preprocessing.ipynb`. Each model is wrapped in the same `Pipeline` (preprocessing + classifier) so hyperparameter search never sees the test set and every fold gets its own freshly-fit imputer, scaler, and encoder.

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !git clone -q https://github.com/AnaNicuesa/loan-approval-prediction.git
    %cd loan-approval-prediction
    %pip install -q -r requirements.txt

In [2]:
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    path = start.resolve()
    for _ in range(4):
        if (path / "src").exists() and (path / "data").exists():
            return path
        path = path.parent
    raise FileNotFoundError("Could not locate project root from " + str(start))


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [3]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from src.train import (
    get_model_specs,
    tune_model,
    save_model,
    build_majority_baseline,
    build_credit_history_only_pipeline,
)
from src.evaluate import compute_metrics
from src.utils import PROCESSED_DATA_DIR, REPORTS_DIR, TARGET, NUMERIC_FEATURES, CATEGORICAL_FEATURES, RANDOM_STATE, MODELS_DIR, timer, ensure_directories

ensure_directories()
pd.set_option("display.max_columns", None)

## 1. Loading the processed split

In [4]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")

feature_columns = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X_train, y_train = train_df[feature_columns], train_df[TARGET]
X_test, y_test = test_df[feature_columns], test_df[TARGET]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Train: (408, 13), Test: (102, 13)


## 2. Baseline models

Before tuning anything, it's worth establishing what "no real modeling effort" looks like on this problem. Two baselines:

- **Majority class baseline**: always predicts the most common label. This is the floor — any model that can't beat it isn't learning anything.
- **`credit_history`-only model**: a plain Logistic Regression using nothing but the one feature that dominated every correlation and importance plot so far. If the full-feature models can't clearly beat this, the extra features (and the extra modeling complexity) aren't earning their keep.

In [5]:
majority_baseline = build_majority_baseline()
majority_baseline.fit(X_train, y_train)

credit_history_baseline = build_credit_history_only_pipeline()
credit_history_baseline.fit(X_train, y_train)

baselines = {
    "majority_baseline": majority_baseline,
    "credit_history_only": credit_history_baseline,
}

baseline_results = []
for name, model in baselines.items():
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    metrics = compute_metrics(y_test, y_pred, y_proba)
    metrics["model"] = name
    baseline_results.append(metrics)

baseline_df = pd.DataFrame(baseline_results).set_index("model")
baseline_df

,accuracy,precision,recall,f1,roc_auc
model,,,,,
majority_baseline,0.715686,0.715686,1.000000,0.834286,0.500000
credit_history_only,0.833333,0.825581,0.972603,0.893082,0.727681


The majority baseline reaches 71-72% accuracy purely from class imbalance, with a ROC AUC of exactly 0.5 -- it has no ranking ability at all, which is precisely why accuracy alone is a misleading metric on this dataset. The `credit_history`-only model already reaches a ROC AUC in the 0.72-0.73 range with a single feature -- whatever the tuned models score above that is the actual value added by the rest of the feature set and the extra modeling effort. That comparison is revisited numerically in the summary at the end of this notebook once the tuned models are in.

## 3. Cross-validation strategy

5-fold stratified cross-validation, optimizing for ROC AUC rather than accuracy — with a 71/29 class split, a model that always predicts "approved" would already score 71% accuracy without learning anything useful, as the majority baseline above confirms directly.

In [6]:
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
model_specs = get_model_specs()
[spec.name for spec in model_specs]

['logistic_regression', 'random_forest', 'xgboost']

## 4. Hyperparameter search

Logistic Regression has a small, well-understood search space so it gets an exhaustive `GridSearchCV`. Random Forest and XGBoost have larger, continuous-valued search spaces where `RandomizedSearchCV` finds competitive configurations in a fraction of the time a full grid would take.

In [7]:
search_results = {}
timings = {}

for spec in model_specs:
    with timer() as t:
        search = tune_model(spec, X_train, y_train, cv=cv_strategy, scoring="roc_auc")
    search_results[spec.name] = search
    timings[spec.name] = t["seconds"]
    print(f"{spec.name}: best CV ROC AUC = {search.best_score_:.4f} ({t['seconds']:.1f}s)")
    print(f"  best params: {search.best_params_}\n")

logistic_regression: best CV ROC AUC = 0.7036 (1.1s)
  best params: {'classifier__C': 0.01, 'classifier__penalty': 'l2', 'classifier__solver': 'lbfgs'}



random_forest: best CV ROC AUC = 0.7508 (5.4s)
  best params: {'classifier__max_depth': 11, 'classifier__max_features': None, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 188}



xgboost: best CV ROC AUC = 0.7495 (1.4s)
  best params: {'classifier__colsample_bytree': 0.9584365199693973, 'classifier__learning_rate': 0.10540104249155915, 'classifier__max_depth': 2, 'classifier__n_estimators': 484, 'classifier__subsample': 0.6911740650167767}



## 5. Comparing tuned models on the held-out test set

In [8]:
test_results = []

for name, search in search_results.items():
    best_pipeline = search.best_estimator_

    with timer() as pred_timer:
        y_pred = best_pipeline.predict(X_test)
        y_proba = best_pipeline.predict_proba(X_test)[:, 1]

    metrics = compute_metrics(y_test, y_pred, y_proba)
    metrics["model"] = name
    metrics["cv_roc_auc"] = search.best_score_
    metrics["training_time"] = timings[name]
    metrics["prediction_time"] = pred_timer["seconds"]
    test_results.append(metrics)

results_df = pd.DataFrame(test_results).set_index("model")
results_df = results_df[["accuracy", "precision", "recall", "f1", "roc_auc", "cv_roc_auc", "training_time", "prediction_time"]]
results_df = results_df.sort_values("roc_auc", ascending=False)
results_df

,accuracy,precision,recall,f1,roc_auc,cv_roc_auc,training_time,prediction_time
model,,,,,,,,
random_forest,0.803922,0.827160,0.917808,0.870130,0.818139,0.750813,5.399633,0.023232
xgboost,0.813725,0.864865,0.876712,0.870748,0.775626,0.749526,1.428624,0.009608
logistic_regression,0.833333,0.825581,0.972603,0.893082,0.745867,0.703582,1.065885,0.006596


Saved to `outputs/reports/model_results.csv` — this is the authoritative report the README results table and the evaluation notebook both read from. The baselines are intentionally kept out of this file since they aren't real candidates for production; they're saved separately below purely for the comparison narrative.

In [9]:
results_df.reset_index().to_csv(REPORTS_DIR / "model_results.csv", index=False)

full_comparison = pd.concat([baseline_df[["accuracy", "precision", "recall", "f1", "roc_auc"]],
                              results_df[["accuracy", "precision", "recall", "f1", "roc_auc"]]])
full_comparison = full_comparison.sort_values("roc_auc", ascending=False)
full_comparison.reset_index().rename(columns={"index": "model"}).to_csv(
    REPORTS_DIR / "baseline_comparison.csv", index=False
)
print(f"Saved report to {(REPORTS_DIR / 'model_results.csv').relative_to(PROJECT_ROOT)}")
print(f"Saved baseline comparison to {(REPORTS_DIR / 'baseline_comparison.csv').relative_to(PROJECT_ROOT)}")
full_comparison

Saved report to outputs/reports/model_results.csv
Saved baseline comparison to outputs/reports/baseline_comparison.csv


,accuracy,precision,recall,f1,roc_auc
model,,,,,
random_forest,0.803922,0.827160,0.917808,0.870130,0.818139
xgboost,0.813725,0.864865,0.876712,0.870748,0.775626
logistic_regression,0.833333,0.825581,0.972603,0.893082,0.745867
credit_history_only,0.833333,0.825581,0.972603,0.893082,0.727681
majority_baseline,0.715686,0.715686,1.000000,0.834286,0.500000


## 6. Selecting and persisting the best model

The model with the highest test-set ROC AUC is saved as `best_model.joblib` — the artifact loaded by the evaluation notebook and the Streamlit app. Every tuned pipeline is also saved individually so any of them can be inspected or swapped in later without retraining. Baselines are excluded from selection by construction — they exist to be beaten, not to be shipped.

In [10]:
best_model_name = results_df["roc_auc"].idxmax()
best_pipeline = search_results[best_model_name].best_estimator_

print(f"Best model: {best_model_name} (test ROC AUC = {results_df.loc[best_model_name, 'roc_auc']:.4f})")

for name, search in search_results.items():
    save_model(search.best_estimator_, name)

save_model(best_pipeline, "best_model")
print(f"Saved all tuned pipelines and best_model.joblib to {MODELS_DIR.relative_to(PROJECT_ROOT)}")

Best model: random_forest (test ROC AUC = 0.8181)


Saved all tuned pipelines and best_model.joblib to models


## 7. Summary

- Logistic Regression, Random Forest, and XGBoost were each tuned with 5-fold cross-validation on ROC AUC.
- `credit_history` dominates in every model, consistent with the correlation seen during EDA.
- **Does the final model actually learn more than `credit_history` alone?** Yes, but the size of that gain depends on which model:

In [11]:
credit_history_auc = baseline_df.loc["credit_history_only", "roc_auc"]
for name in results_df.index:
    gain = results_df.loc[name, "roc_auc"] - credit_history_auc
    print(f"  {name}: ROC AUC {results_df.loc[name, 'roc_auc']:.3f} vs. {credit_history_auc:.3f} baseline ({gain:+.3f})")

  random_forest: ROC AUC 0.818 vs. 0.728 baseline (+0.090)
  xgboost: ROC AUC 0.776 vs. 0.728 baseline (+0.048)
  logistic_regression: ROC AUC 0.746 vs. 0.728 baseline (+0.018)


Random Forest adds a meaningful gain over the single-feature baseline — it's picking up real interaction effects between income, loan amount, and the other applicant attributes. Logistic Regression barely improves on the baseline at all, which makes sense: without engineered interaction terms, a linear model has little room to exploit features beyond the one dominant signal.

That last point is worth making concrete rather than taking on faith: tuned Logistic Regression's accuracy, precision, recall, and F1 on the test set are **identical**, to several decimal places, to the `credit_history`-only baseline's. The only thing that differs is ROC AUC. In other words, at the default 0.5 threshold the extra features don't flip a single classification decision for this model — they only adjust the underlying probability ranking. That is about as direct a demonstration as this dataset can give that a linear model isn't extracting much beyond the one dominant feature, and it's exactly the kind of result that justifies picking Random Forest over the simpler, more explainable linear model here.

The best-performing pipeline is persisted as `models/best_model.joblib` and reused unchanged by `04_Model_Evaluation.ipynb` and the Streamlit app, so the numbers reported there always match what was actually selected here.

> **Potential Interview Questions**
>
> **Q: Why compare against a majority-class baseline at all -- isn't it obviously going to lose?**
> A: The point isn't whether it loses, it's *by how much*, and on what metric. Here it hits 71% accuracy and a ROC AUC of exactly 0.5 -- which is exactly the argument for not trusting accuracy on an imbalanced target. If a stakeholder later asks "why not just approve everyone," this table is the answer.
>
> **Q: The `credit_history`-only model already gets 0.73 ROC AUC. Why not just ship that instead of a Random Forest?**
> A: Because Random Forest adds a real, measurable gain (~0.09 AUC) by combining `credit_history` with income and loan-size interactions, and it's still cheap to train and serve at this data volume. If the gain had been negligible, the simpler single-feature model would be the right call -- simplicity is free until a more complex model earns its keep, which here it does.
>
> **Q: Why GridSearchCV for one model and RandomizedSearchCV for the others?**
> A: Logistic Regression's search space here is 5 discrete values for one hyperparameter -- small enough to search exhaustively. Random Forest and XGBoost have several continuous or wide-ranging hyperparameters where a full grid would be combinatorially expensive for no real benefit over a randomized search with a reasonable budget.
>
> **Q: How do you know the cross-validation folds aren't leaking information from test into train?**
> A: The `ColumnTransformer` (imputation, scaling, encoding) lives inside the `Pipeline` that gets cross-validated, so it's refit on each fold's training data only -- it never sees that fold's validation rows, and it never sees the held-out test set until the very end.
>
> **Q: What would you change if this had to run in production tomorrow?**
> A: Nothing about the modeling choice, but I would not retrain on this exact 563-row split -- I'd want repeated cross-validation or a larger dataset to trust the hyperparameters, and I'd want the baseline comparison re-run any time the feature set changes, not just once here.